In [ ]:
import numpy as np
import xarray as xr

In [ ]:
from conus_biomass import dir_info

dir_info.dir_fia_csvs

In [ ]:
from conus_biomass.process_inputs import restructure_NFI_data

In [ ]:
from conus_biomass.dir_info import dir_processed
from conus_biomass.process_inputs import load_prism
from conus_biomass.settings import STATE_LIST

# Loop through everything for a single state

In [ ]:
ds_combined = restructure_NFI_data.process_single_state(
    state="CA", fpath_out=dir_info.dir_processed + "CA_FIA_plots_and_PRISM.nc"
)

In [ ]:
import matplotlib.pyplot as plt

plt.hexbin(
    x=ds_combined["lon"],
    y=ds_combined["lat"],
    C=ds_combined["biomass_most_recent"],
    vmax=180,
    gridsize=80,
    edgecolors="none",
)
plt.colorbar()

# Loop through all states

In [ ]:
restructure_NFI_data.process_all_states(state_list=STATE_LIST)

In [ ]:
# Running through all Western states on Claire's computer takes ~30 minutes
# Running through ALL states takes ~2.5 hours

for i, state in enumerate(STATE_LIST):

    print("--------------------")
    print("Processing state: " + state)
    cond_data = restructure_NFI_data.process_fia_data(state=state)

    cond_data_recent = restructure_NFI_data.filter_cond_data(cond_data)
    print(len(cond_data_recent))

    # Some checks to add here: are there any STATE_PLOT entries with conflicting lat/lons?
    # if so that is a problem and means we needed to also have a county identifier
    # Not sure whether/how to deal with plots on state boundaries but currently ignoring this

    print("Running through restructuring")
    ds = restructure_NFI_data.run_through_restructuring(cond_data_recent)

    print("Loading PRISM data")
    prism_ds = load_prism.load_prism_data_for_all_plots(ds)

    print("Combining datasets")
    ds_combined = load_prism.combine_two_datasets(ds, prism_ds)

    print("Saving dataset")
    ds_combined.to_netcdf(
        dir_processed + "restructured_FIA/all_prism_months/" + state + "_FIA_plots_and_PRISM.nc"
    )
    print(len(ds_combined))

In [ ]:
for i, state in enumerate(STATE_LIST):

    ds_state = xr.open_dataset(
        dir_processed + "restructured_FIA/all_prism_months/" + state + "_FIA_plots_and_PRISM.nc"
    )

    for var in ["tmean", "ppt", "tdmean", "tmax", "tmin", "vpdmax", "vpdmin"]:
        # print(var)

        ds_state_clim = ds_state.sel(time=ds_state["time.year"].isin(np.arange(1981, 2011)))
        ds_state[var + "_clim_mean"] = ds_state_clim[var].mean(
            dim="time"
        )  # .rename({"plot": "plotid"})
        ds_state[var + "_clim_minseason"] = (
            ds_state_clim[var]
            .groupby("time.season")
            .mean(dim="time")
            .min(dim="season")
            # .rename({"plot": "plotid"})
        )
        ds_state[var + "_clim_maxseason"] = (
            ds_state_clim[var]
            .groupby("time.season")
            .mean(dim="time")
            .max(dim="season")
            # .rename({"plot": "plotid"})
        )

        ds_state[var + "_mean_anom"] = xr.full_like(ds_state["biomass"], fill_value=np.nan)
        ds_state[var + "_minseason_anom"] = xr.full_like(ds_state["biomass"], fill_value=np.nan)
        ds_state[var + "maxseason_anom"] = xr.full_like(ds_state["biomass"], fill_value=np.nan)

        for year in np.arange(1991, 2024):
            ds_state_10yr = ds_state.sel(
                time=ds_state["time.year"].isin(np.arange(year - 10, year))
            )
            ds_state[var + "_mean_anom"].loc[:, ds_state["year"] == year] = (
                ds_state_10yr[var].mean(dim="time") - ds_state[var + "_clim_mean"]
            )
            ds_state[var + "_minseason_anom"].loc[:, ds_state["year"] == year] = (
                ds_state_10yr[var].groupby("time.season").mean(dim="time").min(dim="season")
            ) - ds_state[var + "_clim_minseason"]

            ds_state[var + "maxseason_anom"].loc[:, ds_state["year"] == year] = (
                ds_state_10yr[var].groupby("time.season").mean(dim="time").max(dim="season")
            ) - ds_state[var + "_clim_maxseason"]

    ds_state = ds_state.drop_vars(["tmean", "ppt", "tdmean", "tmax", "tmin", "vpdmax", "vpdmin"])

    ds_state.to_netcdf(dir_processed + "restructured_FIA/" + state + "_FIA_plots_and_PRISM_v6.nc")

# Load everything and look at map

In [ ]:
ds_all = xr.open_mfdataset(dir_processed + "restructured_FIA/*_FIA_plots_and_PRISM_v5.nc")

In [ ]:
ds_all_new = xr.open_mfdataset(dir_processed + "restructured_FIA/*_FIA_plots_and_PRISM_v6.nc")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 9))
plt.hexbin(
    x=ds_all["lon"],
    y=ds_all["lat"],
    C=ds_all["biomass_delta"],
    gridsize=120,
    cmap="BrBG",
    vmin=-25,
    vmax=25,
    edgecolors="none",
)
plt.colorbar()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 9))
plt.hexbin(
    x=ds_all["lon"],
    y=ds_all["lat"],
    C=ds_all_new["biomass_delta"],
    gridsize=120,
    cmap="BrBG",
    mincnt=5,
    vmin=-30,
    vmax=30,
    edgecolors="none",
)
plt.colorbar()

In [ ]:
np.round(ds_all_new["CONDPROP_UNADJ"].sel(year=ds_all["measyear_2"]).values, 1)

In [ ]:
import seaborn as sns

plt.figure(figsize=(30, 6))
sns.boxplot(
    x=np.round(ds_all_new["CONDPROP_UNADJ"].sel(year=ds_all["measyear_2"]).values, 2),
    y=ds_all_new["biomass"].sel(year=ds_all_new["measyear_2"]).values,
)
plt.ylim([0, 500])

In [ ]:
plt.figure(figsize=(15, 9))
plt.hist2d(x=ds_all["lon"], y=ds_all["lat"], bins=130, vmax=10, cmap=plt.cm.viridis)
plt.colorbar()

In [ ]:
plt.figure(figsize=(15, 9))
plt.hexbin(
    x=ds_all["lon"],
    y=ds_all["lat"],
    C=ds_all_new["biomass_most_recent"].where(
        ds_all_new["CONDPROP_UNADJ"].sel(year=ds_all_new["measyear_2"]) > 0.1
    ),
    gridsize=140,
    cmap="viridis",
    mincnt=5,
    vmin=0,
    vmax=160,
    edgecolors="none",
)
plt.colorbar()

In [ ]:
plt.figure(figsize=(15, 9))
plt.hexbin(
    x=ds_all["lon"],
    y=ds_all["lat"],
    C=ds_all["biomass_most_recent"],
    gridsize=130,
    cmap="viridis",
    vmin=0,
    vmax=160,
    edgecolors="none",
)
plt.colorbar()

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.hexbin(
    x=ds_all["lon"],
    y=ds_all["lat"],
    C=(~np.isnan(ds_all["biomass"])).sum(dim="year"),
    gridsize=130,
    cmap=plt.get_cmap("viridis", 6),
    edgecolors="none",
    vmin=0.5,
    vmax=6.5,
)
plt.colorbar()
plt.subplot(1, 2, 2)
plt.hist((~np.isnan(ds_all["biomass"])).sum(dim="year").values, bins=np.arange(-0.5, 8.5, 1))
print(np.nanmax((~np.isnan(ds_all["biomass"])).sum(dim="year").values))
plt.tight_layout()